In [1]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

pdfs = DirectoryLoader(r"G:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\4910-LangChain-Tecnicas-Avancas-de-RAG\documentos", glob="*.pdf", loader_cls=PyPDFLoader).load()


C:\Users\MarcosRibeiro\AppData\Local\Temp\ipykernel_40152\552424176.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
g:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
len(pdfs)

59

In [3]:
from transformers import AutoTokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [5]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer, chunk_size=1250, chunk_overlap=150
)

In [6]:
pedacos = splitter.split_documents(pdfs)

In [7]:
len(pedacos)

59

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="bge-m3:567m")

vector_store = FAISS.from_documents(
    documents=pedacos, embedding=embeddings
)

In [9]:
retriever = vector_store.as_retriever()

In [10]:
from langchain_ollama.llms import OllamaLLM

modelo = OllamaLLM(model="gemma3:4b")

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda usando exclusivamente os conteúdo fornecido. \n\nContexto:\n{contexto}"),
        ("human", "{query}")
    ]
)

In [12]:
from langchain_core.output_parsers import StrOutputParser

cadeia = prompt | modelo | StrOutputParser()

In [13]:
pergunta = "Como fazer um seguro viagem?"

trechos = retriever.invoke(pergunta)
contexto = "\n\n".join(trecho.page_content for trecho in trechos)

cadeia.invoke({"query": pergunta, "contexto": contexto})


'Para fazer um seguro viagem, siga estes passos, com base nas informações fornecidas:\n\n1.  **Verifique a Elegibilidade:** Certifique-se de que você se qualifica para o programa MasterAssist Plus, que é oferecido aos portadores de cartões Mastercard Platinum™ e seus dependentes.\n\n2.  **Emita o Bilhete de Seguro:** Emita um Bilhete de Seguro Viagem através do portal [www.aig.com/Mastercard/pt](http://www.aig.com/Mastercard/pt). Este documento é essencial para acionar a cobertura. O bilhete tem vigência de 12 meses a partir da data de emissão.\n\n3.  **Garanta a Cobertura:** A cobertura se aplica quando a passagem para o transporte público autorizado é paga com seu cartão Mastercard Platinum™ ou com pontos ganhos em um Programa de Recompensas associado ao cartão.\n\n4.  **Entenda os Limites:** Esteja ciente dos limites de cobertura:\n    *   Despesas Médicas e Hospitalares: Até USD 25.000 por Pessoa Elegível.\n    *   Traslado Médico: Até USD 50.000.\n    *   Outros benefícios: Consul

In [16]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "contexto": RunnablePassthrough() | retriever,
        "query": RunnablePassthrough()
    }
    | prompt
    | modelo
    | StrOutputParser()
)